In [ ]:
import time
import torch
from genkit._solvers import sample_flow
from genkit.diffusion import DLPMEps
from genkit.flow_matching import GaussianFlowLinear
from genkit.training import train
from _constants import ALPHA, DEVICE, DIM, DTYPE, N_STEPS, N_TEST, N_TRIALS, TRAIN_KWARGS
from _utils import add_test_vs_true_sample, evaluate_samples, load_alpha_stable_data, make_net, plot_tail_results


In [ ]:
x_train, x_test = load_alpha_stable_data()

train_radius = x_train.reshape(x_train.shape[0], -1).norm(dim=1)


class Proto(GaussianFlowLinear):
    """GFL with a binary endpoint tail flag appended to the vector-field input."""

    def __init__(self, tail_threshold, tail_prob, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.tail_threshold = torch.as_tensor(tail_threshold, device=self._device, dtype=self._fdtype).detach()
        self.tail_prob = float(tail_prob)

    def _tail_flag(self, x):
        radius = x.reshape(x.shape[0], -1).norm(dim=1, keepdim=True)
        return (radius >= self.tail_threshold.to(device=x.device, dtype=x.dtype)).to(dtype=x.dtype)

    def _append_flag(self, x, flag):
        flag = flag.to(device=x.device, dtype=x.dtype).reshape(x.shape[0], 1)
        return torch.cat([x.reshape(x.shape[0], -1), flag], dim=1)

    def _loss(self, x, z=None, t=None):
        x_0, x_1, t = self._latent(x_1=x, x_0=z, t=t)
        t_data = self._expand_batch_scalar(t, x_0)
        x_t = (1.0 - t_data) * x_0 + t_data * x_1
        v_t = x_1 - x_0
        v_t_hat = self._net(self._append_flag(x_t, self._tail_flag(x_1)), t).reshape_as(v_t)
        return self._loss_fn(v_t_hat, v_t, t)

    @torch.inference_mode()
    def _sample(self, n_samples=None, return_trajectory=True, sample_source=None, chunk_size=None):
        if chunk_size is not None:
            raise ValueError("Proto does not support chunked sampling.")

        self._net.eval()
        n_samples, x = self._resolve_sample_source(n_samples, sample_source)
        flag = (torch.rand(n_samples, 1, device=self._device, dtype=self._fdtype) < self.tail_prob).to(self._fdtype)

        def field(x_t, t):
            return self._net(self._append_flag(x_t, flag), t).reshape_as(x_t)

        if self._sampler == "adaptive_heun":
            return sample_flow(
                field,
                x,
                sampler=self._sampler,
                t_min=self._t_min,
                t_max=self._t_max,
                atol=self._atol,
                rtol=self._rtol,
                h_init=self._h_init,
                h_min=self._h_min,
                h_max=self._h_max,
                return_trajectory=return_trajectory,
            )
        return sample_flow(field, x, self._sample_timesteps, sampler=self._sampler, return_trajectory=return_trajectory)


In [ ]:
rows = []
for trial in range(1, N_TRIALS + 1):
    torch.manual_seed(trial - 1)
    print(f"Trial {trial} / {N_TRIALS}: ", end="")

    start_time = time.time()
    dlpm = DLPMEps(
        net=make_net(DIM),
        alpha=ALPHA,
        dim=DIM,
        n_steps=N_STEPS,
        n_trial_A=1,
        n_trial_G=1,
        reduce_type="mean",
        device=DEVICE,
    )
    dlpm, _ = train(dlpm, x_train, **TRAIN_KWARGS)
    rows.append(evaluate_samples("DLPM", trial, x_test, dlpm.sample(N_TEST)))
    print(f"'DLPM' done in {time.time() - start_time:.2f} s", end="", flush=True)

    start_time = time.time()
    gfl = GaussianFlowLinear(
        net=make_net(DIM),
        dim=DIM,
        n_steps=N_STEPS,
        sample_steps=N_STEPS,
        device=DEVICE,
    )
    gfl, _ = train(gfl, x_train, **TRAIN_KWARGS)
    rows.append(evaluate_samples("GFL", trial, x_test, gfl.sample(N_TEST)))
    print(f", 'GFL' done in {time.time() - start_time:.2f} s", end="", flush=True)

    tail_quantile = 0.9
    start_time = time.time()
    tail_threshold = torch.quantile(train_radius, tail_quantile).detach()
    proto_gfl = Proto(
        net=make_net(DIM, input_dim=DIM + 1, output_dim=DIM),
        tail_threshold=tail_threshold,
        tail_prob=(train_radius >= tail_threshold).to(DTYPE).mean().item(),
        dim=DIM,
        n_steps=N_STEPS,
        sample_steps=N_STEPS,
        device=DEVICE,
    )
    proto_gfl, _ = train(proto_gfl, x_train, **TRAIN_KWARGS)
    rows.append(evaluate_samples(f"Tail-flag GFL (q={tail_quantile:.1f})", trial, x_test, proto_gfl.sample(N_TEST)))
    print(f", 'Tail-flag GFL (q={tail_quantile:.1f})' done in {time.time() - start_time:.2f} s", end="", flush=True)

    print(" .", flush=True)

add_test_vs_true_sample(rows, x_test)


In [ ]:
fig, ax = plot_tail_results(rows, title="Tail-conditioned GFL")
fig.show()
